# Marketplace Safety - Prioritization of Suspicious Listings and Messages

**Background**  
This project was carried out on behalf of the analytics team at a company that operates a 
marketplace app, similar to Blocket. The platform is used daily by a large number of users, the vast majority of whom are genuine. However, every week a small proportion of problematic activity occurs: scam listings, spam, suspicious accounts acting quickly, and attempts to move conversations off the platform.

**Problem**  
The company's Trust & Safety team manually reviews and handles suspicious content. The problem is that the volume is too large for the team to review everything in time. Management is therefore requesting a decision-support tool that helps the team prioritize what to review first.

**Goal**  
The goal is to deliver a solution that:
- performs reasonably well on new, unseen data
- can be run regularly in production
- can be explained to non-technical stakeholders

The focus is on prioritization and decision support.

**Stakeholder**  
The COO sees 2 costs:
- the time required to check the cases
- the missed scams

Kolumnnamn | Förklaring |
|---|---|
| `id` | Unikt identifieringsnummer för varje post |
| `day` | Dagen då händelsen inträffade |
| `event_type` | Typ av händelse (t.ex. visning, meddelande, anmälan) |
| `category` | Produktkategori för annonsen |
| `region` | Geografiskt område där annonsen publicerades |
| `device` | Enhetstyp som användes (t.ex. mobil, dator) |
| `account_age_days` | Antal dagar sedan kontot skapades |
| `num_prev_listings` | Antal tidigare annonser från samma användare |
| `prev_reports_30d` | Antal gånger användaren anmälts de senaste 30 dagarna |
| `verification_level` | Verifieringsnivå för kontot |
| `price` | Annonsens pris |
| `num_images` | Antal bilder i annonsen |
| `message_length` | Längden på meddelandet i annonsen |
| `contains_off_platform` | Om användaren försökt flytta konversationen utanför plattformen |
| `urgency_words` | Om annonsen innehåller ord som skapar artificiell brådska |
| `payment_attempt` | Om ett betalningsförsök har skett |
| `time_to_first_response_min` | Tid i minuter till första svar |
| `is_suspicious` | Målvariabel — om annonsen är misstänkt (1) eller inte (0) |

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, 
    cross_val_predict,
    cross_val_score, 
    GridSearchCV, 
    cross_validate
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from joblib import dump, load

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier


from sklearn.metrics import (
    accuracy_score, 
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(palette="rocket")

### Data understanding & EDA

- Showing dataset size, data types and statistics
- Checking for missing values 
- Showing target distribution of the target
- Checking correlation among numeric variables

In [ ]:
marketplace_data = pd.read_csv("../Data/historical_data.csv")

print(marketplace_data.shape)
marketplace_data.head()

In [ ]:
marketplace_data.describe().T

In [ ]:
marketplace_data.dtypes

In [ ]:
print("Total missing values:\n\n", marketplace_data.isna().sum())
print("\n")
print("Percentage of missing values:\n\n", round(marketplace_data.isnull().sum() / len(marketplace_data) * 100),3)

In [ ]:
X_full = marketplace_data.drop(["is_suspicious"], axis=1)
y_full = marketplace_data["is_suspicious"]

print("X:", X_full.shape, "\ny:", y_full.shape)

In [ ]:
y_distribution = y_full.value_counts(normalize=True)

fig1, ax = plt.subplots(figsize=(8,7))
suspocious_distribution = ax.bar(y_distribution.index, y_distribution.values)

for bar in suspocious_distribution:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.005,
            f'{height:.1%}', ha='center', va='bottom')

ax.set_title("Class distribution of Suspicious label", fontsize=14)
ax.set_xlabel("Suspicious (0 = No, 1 = Yes)")
ax.set_ylabel("Proportion")
ax.set_xticks([0, 1])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

plt.savefig("../Img/1_suspicious_share")
plt.show()

In [ ]:
correlation_with_target = marketplace_data.copy()

correlation_with_target = (
    correlation_with_target.drop(
        ["event_type", "category", "region", "device"], axis=1)
        .corr()["is_suspicious"]
        .sort_values(ascending=False)
)

display(correlation_with_target)

### Train/test + preprocessing

- Created a train/test split of the data
- Built a pipeline where preprocessing is done in a way that prevents test data from influencing training (avoid leakage).
- For classification: use stratified split so classes are distributed reasonably.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=42,
    stratify=y_full
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("\n")
print(f"Classification train: \n{y_train.value_counts(normalize=True)}")
print("\n")
print(f"Classification test: \n{y_test.value_counts(normalize=True)}")

In [ ]:
def engineer_features(X):
    X = X.copy()
    X["report_rate"] = X["prev_reports_30d"] / (X["num_prev_listings"] + 1)
    X["risk_score"]  = X["contains_off_platform"] + X["urgency_words"] + X["payment_attempt"]
    X["new_and_reported"] = ((X["account_age_days"] < 30) & (X["prev_reports_30d"] > 0)).astype(int)

    return X

feature_engineer = FunctionTransformer(engineer_features)

In [ ]:
numeric_features_base = [
    "account_age_days", "num_prev_listings", "prev_reports_30d",
    "price", "num_images", "message_length", "verification_level",
    "time_to_first_response_min", "payment_attempt",
    "contains_off_platform", "urgency_words"
]

numeric_features_engineered = numeric_features_base + ["report_rate", "risk_score", "new_and_reported"]

categorical_features = ["event_type", "category", "region", "device"]

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler",  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [ ]:
preprocess_none = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features_base),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

preprocess_engineered = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features_engineered),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [ ]:
basic_features_pipeline = Pipeline(steps=[
    ("preprocess", preprocess_none),
    ("classifier", RandomForestClassifier(
        class_weight="balanced", n_estimators=200,
        min_samples_leaf=5, max_depth=10, random_state=42))
])

with_engineered_features_pipeline = Pipeline(steps=[
    ("features",   FunctionTransformer(engineer_features)),
    ("preprocess", preprocess_engineered),
    ("classifier", RandomForestClassifier(
        class_weight="balanced", n_estimators=200,
        min_samples_leaf=5, max_depth=10, random_state=42))
])

In [ ]:
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results_base = cross_validate(basic_features_pipeline, X_train, y_train, cv=5, scoring=scoring)
results_engineered = cross_validate(with_engineered_features_pipeline, X_train, y_train, cv=5, scoring=scoring)

rows = []
for metric in scoring:
    base = results_base[f"test_{metric}"].mean()
    eng  = results_engineered[f"test_{metric}"].mean()
    rows.append({"Metric": metric, "Base": round(base, 4), "Engineered": round(eng, 4), "Difference": round(eng - base, 4)})

pd.DataFrame(rows).set_index("Metric")

3) Modeling and comparison

- Create a baseline.
- Train at least two additional models (at least 3 total including baseline).
- Evaluate with a reasonable method (e.g. cross-validation on train or a clear validation setup).
- Choose a metric and justify the choice based on your requirements card.
(Example models: LogisticRegression, DecisionTree, RandomForest, GradientBoosting…)

In [ ]:
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced", min_samples_leaf=5, max_depth=10),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=42, class_weight="balanced")
}

In [ ]:
# Building one pipeline for each model
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline(steps=[
        ("feature_engineer", feature_engineer),
        ("preprocessor", preprocess_engineered),
        ("classifier", model)
    ])

# Evaluating each model with cross_val_predict and showing the confusion matrix for each model
rows = []

fig2, axes = plt.subplots(2, 3, figsize=(18,10))
axes_flat = axes.flatten()
axes_flat[5].set_visible(False)

for idx, (name, pipeline) in enumerate (pipelines.items()):
    y_pred = cross_val_predict(pipeline, X_train, y_train, cv=5, n_jobs=-1)

    y_proba = cross_val_predict(
        pipeline,
        X_train,
        y_train,
        cv=5,
        n_jobs=-1,
        method="predict_proba"
    )[:, 1]

    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_train, y_pred),
        "precision": precision_score(y_train, y_pred, zero_division=0),
        "recall": recall_score(y_train, y_pred, zero_division=0),
        "f1": f1_score(y_train, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_train, y_proba)
    })

    cm = confusion_matrix(y_train, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Legit", "Suspicious"])
    disp.plot(ax=axes_flat[idx], colorbar=False, cmap="rocket")
    axes_flat[idx].grid(False)
    axes_flat[idx].set_title(name, size=15)

plt.tight_layout()
plt.savefig("../Img/fig2_conf_matrix_all_models")
plt.show()

results_cv = pd.DataFrame(rows).sort_values("f1", ascending=False)
display(results_cv)

The models' relatively low F1 score should be interpreted in the context of the dataset's characteristics. The data is synthetic so the boundaries between suspicious and legitimate listings are less sharp than in real-world data.  

The class imbalance makes F1 harder to optimize but the ROC AUC of ~0.74 indicates that 2 models have genuine discriminative ability, correctly ranking a suspicious listing above a legitimate one ~74% of the time.  

4) Select and optimize ONE model (hyperparameter tuning)

- Select a "final" model based on the comparison.
- Tune the selected model (small grid, at least 1–2 parameters).
- Briefly explain what you optimized and why (connect to the requirements card).

5) Threshold / prioritization (linked to the requirements card)
- You must decide how the model will be used in practice. Choose a strategy:

    A) Threshold decision — flag as suspicious if probability ≥ t. Justify t based on the requirements card and show consequences (FP/FN or precision/recall) 
    B) Top-X prioritization — flag the X% highest risk (e.g. top 5% or top 50 per day). Justify X based on the requirements card and show consequences.

You must clearly show how your choice affects FP/FN and why it suits your stakeholder.

6) Deploy test: new data (Tuesday course week 6)
When you receive new_data.csv you should:

- use your locked pipeline
- create predictions and a priority list